Checagem 1 — Consistência entre pessoas e a soma de mortos/feridos/ilesos/ignorados

In [0]:
df_silver = spark.table("prf_acidentes1.silver.acidentes_limpo")
fato_acidente = spark.table("prf_acidentes1.gold.fato_acidente")

print("Linhas Silver:", df_silver.count())
print("Linhas Gold (fato_acidente):", fato_acidente.count())

Linhas Silver: 213451
Linhas Gold (fato_acidente): 213451


In [0]:
from pyspark.sql import functions as F

df_check1 = (
    df_silver
    .withColumn(
        "soma_calculada",
        F.col("mortos") + F.col("feridos_leves") + F.col("feridos_graves") + F.col("ilesos") + F.col("ignorados")
    )
    .withColumn("diferenca", F.col("pessoas") - F.col("soma_calculada"))
)

# Quantas linhas batem exatamente?
total = df_check1.count()
batem = df_check1.filter(F.col("diferenca") == 0).count()
nao_batem = total - batem

print(f"Total de linhas: {total}")
print(f"Linhas onde pessoas = soma dos estados: {batem}")
print(f"Linhas com divergência: {nao_batem}")

# Mostrar alguns exemplos de divergência, se houver
df_check1.filter(F.col("diferenca") != 0).select(
    "id", "pessoas", "mortos", "feridos_leves", "feridos_graves", "ilesos", "ignorados", "soma_calculada", "diferenca"
).show(20)

Total de linhas: 213451
Linhas onde pessoas = soma dos estados: 201977
Linhas com divergência: 11474
+------+-------+------+-------------+--------------+------+---------+--------------+---------+
|    id|pessoas|mortos|feridos_leves|feridos_graves|ilesos|ignorados|soma_calculada|diferenca|
+------+-------+------+-------------+--------------+------+---------+--------------+---------+
|496590|      2|     0|            0|             1|     0|        2|             3|       -1|
|496610|      2|     0|            0|             0|     1|        2|             3|       -1|
|496795|      3|     0|            1|             0|     1|        3|             5|       -2|
|496818|      3|     1|            0|             0|     1|        2|             4|       -1|
|496834|      4|     1|            1|             1|     0|        2|             5|       -1|
|496862|      9|     1|            2|             1|     4|        2|            10|       -1|
|496901|      2|     0|            0|       

In [0]:
from pyspark.sql import functions as F

# Testar se a divergência segue o padrão: diferenca = 1 - ignorados
df_check1_padrao = df_check1.withColumn(
    "segue_padrao", F.col("diferenca") == (1 - F.col("ignorados"))
)

total_divergentes = df_check1_padrao.filter(F.col("diferenca") != 0).count()
seguem_padrao = df_check1_padrao.filter((F.col("diferenca") != 0) & (F.col("segue_padrao") == True)).count()

print(f"Total de linhas divergentes: {total_divergentes}")
print(f"Linhas que seguem o padrão (diferença = 1 - ignorados): {seguem_padrao}")

Total de linhas divergentes: 11474
Linhas que seguem o padrão (diferença = 1 - ignorados): 10368


In [0]:
from pyspark.sql import functions as F

# Isolar as linhas divergentes que NÃO seguem o padrão "diferenca = 1 - ignorados"
df_resto = df_check1_padrao.filter((F.col("diferenca") != 0) & (F.col("segue_padrao") == False))

print("Total de linhas no grupo residual:", df_resto.count())

df_resto.select(
    "id", "pessoas", "mortos", "feridos_leves", "feridos_graves", "ilesos", "ignorados", "soma_calculada", "diferenca"
).show(20)

# Ver a distribuição dos valores de "diferenca" nesse grupo residual
df_resto.groupBy("diferenca").count().orderBy("diferenca").show(30)

Total de linhas no grupo residual: 1106
+------+-------+------+-------------+--------------+------+---------+--------------+---------+
|    id|pessoas|mortos|feridos_leves|feridos_graves|ilesos|ignorados|soma_calculada|diferenca|
+------+-------+------+-------------+--------------+------+---------+--------------+---------+
|497248|      4|     0|            0|             0|     1|        5|             6|       -2|
|497291|      6|     1|            0|             3|     0|        3|             7|       -1|
|497411|      5|     1|            0|             0|     2|        3|             6|       -1|
|498033|      4|     1|            0|             0|     1|        3|             5|       -1|
|498066|      6|     0|            2|             0|     1|        4|             7|       -1|
|498307|      4|     0|            1|             0|     1|       15|            17|      -13|
|498816|      3|     1|            0|             0|     0|        3|             4|       -1|
|498951|  

Checagem 2 — Validação de domínio: uf e br

In [0]:
from pyspark.sql import functions as F

# UFs válidas do Brasil
ufs_validas = ["AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"]

df_silver.groupBy("uf").count().orderBy(F.desc("count")).show(30)

print("--- BR fora do intervalo plausível (ex: <1 ou >495) ---")
df_silver.filter((F.col("br") < 1) | (F.col("br") > 495)).select("br").distinct().show()

+---+-----+
| uf|count|
+---+-----+
| MG|27873|
| SC|24366|
| PR|22316|
| RJ|18392|
| RS|15041|
| SP|14320|
| BA|12014|
| GO| 9637|
| PE| 9244|
| MT| 7507|
| ES| 7278|
| PB| 5462|
| MS| 5183|
| RN| 4621|
| RO| 4335|
| PI| 4283|
| CE| 4165|
| MA| 3563|
| DF| 3044|
| PA| 2986|
| TO| 2040|
| AL| 1961|
| SE| 1723|
| AC|  792|
| AP|  472|
| RR|  421|
| AM|  412|
+---+-----+

--- BR fora do intervalo plausível (ex: <1 ou >495) ---
+---+
| br|
+---+
|498|
|  0|
+---+



In [0]:
from pyspark.sql import functions as F

df_silver.filter(F.col("br") == 0).select("id", "uf", "br", "km", "municipio").show(10)
print("Total com br = 0:", df_silver.filter(F.col("br") == 0).count())

df_silver.filter(F.col("br") == 498).select("id", "uf", "br", "km", "municipio").show(10)
print("Total com br = 498:", df_silver.filter(F.col("br") == 498).count())

+------+---+---+---+----------------+
|    id| uf| br| km|       municipio|
+------+---+---+---+----------------+
|499508| DF|  0|0.0|        BRASILIA|
|505418| DF|  0|0.0|        BRASILIA|
|509819| MT|  0|0.0|   VARZEA GRANDE|
|511949| MS|  0|0.0|    CAMPO GRANDE|
|516377| DF|  0|0.0|        BRASILIA|
|525234| PR|  0|0.0|    PONTA GROSSA|
|527565| BA|  0|0.0|FEIRA DE SANTANA|
|529454| BA|  0|0.0|         SANTANA|
|530648| MS|  0|0.0|         NAVIRAI|
|534040| RO|  0|0.0|         VILHENA|
+------+---+---+---+----------------+
only showing top 10 rows
Total com br = 0: 513
+------+---+---+----+---------+
|    id| uf| br|  km|municipio|
+------+---+---+----+---------+
|550685| BA|498|11.1|ITAMARAJU|
+------+---+---+----+---------+

Total com br = 498: 1


Checagem 3 — Verificar duplicatas na Gold (fato_acidente) após os joins

In [0]:
total_fato = fato_acidente.count()
ids_distintos_fato = fato_acidente.select("id_acidente").distinct().count()

print(f"Total de linhas na fato_acidente: {total_fato}")
print(f"IDs de acidente distintos: {ids_distintos_fato}")
print(f"Diferença: {total_fato - ids_distintos_fato}")

Total de linhas na fato_acidente: 213451
IDs de acidente distintos: 213451
Diferença: 0
